In [ ]:
!pip install -q yfinance xgboost ta joblib transformers torch

print("✅ All libraries installed successfully.")

✅ All libraries installed successfully.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import io
import re
import sys
import json
import numpy as np
import pandas as pd
import yfinance as yf
import ta
import joblib
import torch

from google.colab import files
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from IPython.display import display, HTML

print("✅ All imports loaded successfully.")

✅ All imports loaded successfully.


In [ ]:
# Known Nifty 50 symbols → Yahoo Finance suffix mapping
KNOWN_STOCKS = {
    "ADANIENT":    "ADANIENT.NS",    "ADANIPORTS":  "ADANIPORTS.NS",
    "APOLLOHOSP":  "APOLLOHOSP.NS",  "ASIANPAINT":  "ASIANPAINT.NS",
    "AXISBANK":    "AXISBANK.NS",     "BAJAJ-AUTO":  "BAJAJ-AUTO.NS",
    "BAJFINANCE":  "BAJFINANCE.NS",   "BAJAJFINSV":  "BAJAJFINSV.NS",
    "BEL":         "BEL.NS",          "BHARTIARTL":  "BHARTIARTL.NS",
    "CIPLA":       "CIPLA.NS",        "COALINDIA":   "COALINDIA.NS",
    "DRREDDY":     "DRREDDY.NS",      "EICHERMOT":   "EICHERMOT.NS",
    "ETERNAL":     "ETERNAL.NS",      "GRASIM":      "GRASIM.NS",
    "HCLTECH":     "HCLTECH.NS",      "HDFCBANK":    "HDFCBANK.NS",
    "HDFCLIFE":    "HDFCLIFE.NS",     "HEROMOTOCO":  "HEROMOTOCO.NS",
    "HINDALCO":    "HINDALCO.NS",     "HINDUNILVR":  "HINDUNILVR.NS",
    "ICICIBANK":   "ICICIBANK.NS",    "INDUSINDBK":  "INDUSINDBK.NS",
    "INFY":        "INFY.NS",         "ITC":         "ITC.NS",
    "JIOFIN":      "JIOFIN.NS",       "JSWSTEEL":    "JSWSTEEL.NS",
    "KOTAKBANK":   "KOTAKBANK.NS",    "LT":          "LT.NS",
    "M&M":         "M&M.NS",          "MARUTI":      "MARUTI.NS",
    "NESTLEIND":   "NESTLEIND.NS",    "NTPC":        "NTPC.NS",
    "ONGC":        "ONGC.NS",         "POWERGRID":   "POWERGRID.NS",
    "RELIANCE":    "RELIANCE.NS",     "SBILIFE":     "SBILIFE.NS",
    "SBIN":        "SBIN.NS",         "SHRIRAMFIN":  "SHRIRAMFIN.NS",
    "SUNPHARMA":   "SUNPHARMA.NS",    "TATACONSUM":  "TATACONSUM.NS",
    "TATAMOTORS":  "TATAMOTORS.NS",   "TATASTEEL":   "TATASTEEL.NS",
    "TCS":         "TCS.NS",          "TECHM":       "TECHM.NS",
    "TITAN":       "TITAN.NS",        "TRENT":       "TRENT.NS",
    "ULTRACEMCO":  "ULTRACEMCO.NS",   "WIPRO":       "WIPRO.NS",
}

# Recommendation thresholds
BUY_THRESHOLD  = 0.70
SELL_THRESHOLD = 0.40

# Hybrid weights
WEIGHT_MODEL     = 0.70
WEIGHT_SENTIMENT = 0.30

# FinBERT model name
FINBERT_MODEL = "ProsusAI/finbert"

print("✅ Configuration loaded.")
print(f"   BUY  threshold : {BUY_THRESHOLD}")
print(f"   SELL threshold : {SELL_THRESHOLD}")
print(f"   Model weight   : {WEIGHT_MODEL}")
print(f"   Sentiment weight: {WEIGHT_SENTIMENT}")

✅ Configuration loaded.
   BUY  threshold : 0.7
   SELL threshold : 0.4
   Model weight   : 0.7
   Sentiment weight: 0.3


In [ ]:
def create_stationary_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recreates all features used during training.
    Input df must have columns: Date, Open, High, Low, Close, Volume
    """
    df = df.sort_values("Date").copy()

    close  = df["Close"]
    open_  = df["Open"]
    high   = df["High"]
    low    = df["Low"]
    volume = df["Volume"]

    prev_close = close.shift(1)

    # ------------------------------------------
    # Target columns (kept for shape consistency)
    # ------------------------------------------
    df["Target_Close_5D"]   = close.shift(-5)
    df["Target_Return_5D"]  = (df["Target_Close_5D"] / close) - 1.0
    df["Target_Direction"]  = (df["Target_Return_5D"] > 0.0).astype(int)

    # ------------------------------------------
    # Return features
    # ------------------------------------------
    df["Ret_1"]     = close.pct_change(1)
    df["Ret_3"]     = close.pct_change(3)
    df["Ret_5"]     = close.pct_change(5)
    df["Ret_10"]    = close.pct_change(10)
    df["Ret_20"]    = close.pct_change(20)
    df["LogRet_1"]  = np.log(close / prev_close)

    # Volume change features
    df["Vol_Chg_1"] = volume.pct_change(1)
    df["Vol_Chg_5"] = volume.pct_change(5)

    # Candle / structure features
    df["Gap_Return"]        = open_ / prev_close - 1.0
    df["Intraday_Return"]   = close / open_ - 1.0
    df["Range_Pct"]         = (high - low) / close
    df["Body_Pct"]          = (close - open_) / open_
    df["Upper_Shadow_Pct"]  = (high - np.maximum(open_, close)) / close
    df["Lower_Shadow_Pct"]  = (np.minimum(open_, close) - low) / close

    # ------------------------------------------
    # Lag features
    # ------------------------------------------
    for i in range(1, 11):
        df[f"Ret_Lag_{i}"]     = df["Ret_1"].shift(i)
        df[f"Vol_Chg_Lag_{i}"] = df["Vol_Chg_1"].shift(i)
        df[f"Range_Lag_{i}"]   = df["Range_Pct"].shift(i)

    # ------------------------------------------
    # Rolling features
    # ------------------------------------------
    for w in [5, 10, 20, 50]:
        price_mean  = close.rolling(w).mean()
        price_std   = close.rolling(w).std()
        vol_mean    = volume.rolling(w).mean()
        vol_std     = volume.rolling(w).std()
        ret_mean    = df["Ret_1"].rolling(w).mean()
        ret_std     = df["Ret_1"].rolling(w).std()
        rolling_high = high.rolling(w).max()
        rolling_low  = low.rolling(w).min()

        df[f"Mom_{w}"]       = close.pct_change(w)
        df[f"Ret_Mean_{w}"]  = ret_mean
        df[f"Ret_Std_{w}"]   = ret_std
        df[f"Price_Z_{w}"]   = (close - price_mean) / price_std
        df[f"Dist_SMA_{w}"]  = close / price_mean - 1.0
        df[f"Dist_High_{w}"] = close / rolling_high - 1.0
        df[f"Dist_Low_{w}"]  = close / rolling_low - 1.0
        df[f"Vol_Z_{w}"]     = (volume - vol_mean) / vol_std
        df[f"Vol_Ratio_{w}"] = volume / vol_mean - 1.0

    # ------------------------------------------
    # Technical indicators
    # ------------------------------------------
    rsi = ta.momentum.RSIIndicator(close=close, window=14).rsi()
    df["RSI_14_N"] = (rsi - 50.0) / 50.0

    macd = ta.trend.MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    df["MACD_N"]        = macd.macd()        / close
    df["MACD_SIGNAL_N"] = macd.macd_signal() / close
    df["MACD_DIFF_N"]   = macd.macd_diff()   / close

    bb = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
    df["BB_PBAND"]   = bb.bollinger_pband()
    df["BB_WBAND_N"] = bb.bollinger_wband() / 100.0

    atr = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14)
    df["ATR_N"] = atr.average_true_range() / close

    stoch = ta.momentum.StochasticOscillator(
        high=high, low=low, close=close, window=14, smooth_window=3
    )
    df["STOCH_K_N"] = stoch.stoch()        / 100.0
    df["STOCH_D_N"] = stoch.stoch_signal() / 100.0

    adx = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14)
    df["ADX_N"] = adx.adx() / 100.0

    mfi = ta.volume.MFIIndicator(high=high, low=low, close=close, volume=volume, window=14)
    df["MFI_N"] = mfi.money_flow_index() / 100.0

    obv = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()
    df["OBV_Slope_10"] = obv.diff(10) / (volume.rolling(20).mean() * 10)

    # ------------------------------------------
    # Calendar features (cyclical)
    # ------------------------------------------
    dow   = df["Date"].dt.dayofweek
    month = df["Date"].dt.month

    df["DOW_SIN"]   = np.sin(2 * np.pi * dow / 7)
    df["DOW_COS"]   = np.cos(2 * np.pi * dow / 7)
    df["MONTH_SIN"] = np.sin(2 * np.pi * (month - 1) / 12)
    df["MONTH_COS"] = np.cos(2 * np.pi * (month - 1) / 12)

    return df


def build_feature_list() -> list:
    """Returns the exact ordered feature list used during training."""
    features = [
        "Stock_ID",
        "Ret_1", "Ret_3", "Ret_5", "Ret_10", "Ret_20",
        "LogRet_1",
        "Vol_Chg_1", "Vol_Chg_5",
        "Gap_Return", "Intraday_Return", "Range_Pct",
        "Body_Pct", "Upper_Shadow_Pct", "Lower_Shadow_Pct",
        "RSI_14_N",
        "MACD_N", "MACD_SIGNAL_N", "MACD_DIFF_N",
        "BB_PBAND", "BB_WBAND_N",
        "ATR_N",
        "STOCH_K_N", "STOCH_D_N",
        "ADX_N", "MFI_N",
        "OBV_Slope_10",
        "DOW_SIN", "DOW_COS", "MONTH_SIN", "MONTH_COS",
    ]
    for i in range(1, 11):
        features += [f"Ret_Lag_{i}", f"Vol_Chg_Lag_{i}", f"Range_Lag_{i}"]
    for w in [5, 10, 20, 50]:
        features += [
            f"Mom_{w}", f"Ret_Mean_{w}", f"Ret_Std_{w}",
            f"Price_Z_{w}", f"Dist_SMA_{w}", f"Dist_High_{w}", f"Dist_Low_{w}",
            f"Vol_Z_{w}", f"Vol_Ratio_{w}",
        ]
    return features


print("✅ Feature engineering functions defined.")

✅ Feature engineering functions defined.


In [ ]:
def detect_stock_symbol(filename: str) -> tuple[str, str]:
    """
    Detects Yahoo Finance ticker from uploaded CSV filename.

    Examples:
        RELIANCE.csv  → ('RELIANCE',  'RELIANCE.NS')
        TCS.csv       → ('TCS',       'TCS.NS')
        INFY_data.csv → ('INFY',      'INFY.NS')

    Returns:
        (raw_symbol, yahoo_ticker)

    Raises:
        ValueError if symbol cannot be resolved.
    """
    # Strip path and extension, uppercase
    base = os.path.splitext(os.path.basename(filename))[0].upper()

    # Strip trailing _data / _historical / -data etc.
    base_clean = re.split(r"[_\-\s]", base)[0]

    # Direct lookup
    if base_clean in KNOWN_STOCKS:
        return base_clean, KNOWN_STOCKS[base_clean]

    # Fuzzy: try full base name
    if base in KNOWN_STOCKS:
        return base, KNOWN_STOCKS[base]

    # Try appending .NS directly and see if it's in our known map values
    candidate = base_clean + ".NS"
    if candidate in KNOWN_STOCKS.values():
        return base_clean, candidate

    raise ValueError(
        f"❌ Cannot resolve stock symbol from filename '{filename}'.\n"
        f"   Detected base: '{base_clean}'.\n"
        f"   Supported symbols: {sorted(KNOWN_STOCKS.keys())}"
    )


def validate_csv_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Validates and normalises uploaded CSV.
    Accepts various column name styles (case-insensitive).

    Required columns: Date, Open, High, Low, Close, Volume

    Returns cleaned DataFrame.
    Raises ValueError on validation failure.
    """
    # Normalise column names → title case
    col_map = {c: c.strip().title() for c in df.columns}
    df = df.rename(columns=col_map)

    required = ["Date", "Open", "High", "Low", "Close", "Volume"]
    missing  = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"❌ CSV is missing required columns: {missing}\n"
            f"   Found columns: {list(df.columns)}"
        )

    # Parse dates
    try:
        df["Date"] = pd.to_datetime(df["Date"])
        df["Date"] = df["Date"].dt.tz_localize(None)
    except Exception as e:
        raise ValueError(f"❌ Cannot parse 'Date' column: {e}")

    # Coerce numeric columns
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows with NaN in OHLCV
    before = len(df)
    df.dropna(subset=["Open", "High", "Low", "Close", "Volume"], inplace=True)
    after = len(df)

    if after == 0:
        raise ValueError("❌ CSV has no valid numeric OHLCV rows after cleaning.")

    if before != after:
        print(f"   ⚠️  Dropped {before - after} rows with invalid OHLCV values.")

    if after < 60:
        raise ValueError(
            f"❌ Insufficient data: {after} rows after cleaning. "
            "Minimum 60 rows required for indicator calculation."
        )

    df = df.sort_values("Date").reset_index(drop=True)
    return df


def safe_get_current_price(yahoo_ticker: str, fallback_close: float) -> float:
    """
    Fetches current price from Yahoo Finance.
    Falls back to last CSV close price on failure.
    """
    try:
        ticker_obj = yf.Ticker(yahoo_ticker)
        info = ticker_obj.fast_info
        price = float(info.last_price)
        if np.isnan(price) or price <= 0:
            raise ValueError("Invalid price from API")
        return price
    except Exception:
        return fallback_close


print("✅ Utility helper functions defined.")

✅ Utility helper functions defined.


In [ ]:
class FinBERTAnalyzer:
    """
    Wraps ProsusAI/finbert for financial headline sentiment.
    Sentiment labels: positive (+1), neutral (0), negative (-1)
    """

    def __init__(self, model_name: str = FINBERT_MODEL):
        print(f"   Loading FinBERT from '{model_name}' ...")
        self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

        # FinBERT label order: positive=0, negative=1, neutral=2
        self.label_map = {0: "positive", 1: "negative", 2: "neutral"}
        self.score_map = {"positive": +1.0, "negative": -1.0, "neutral": 0.0}
        print(f"   FinBERT loaded on device: {self.device}")

    def analyze_headline(self, headline: str) -> dict:
        """
        Analyzes a single headline.

        Returns:
            {
                "headline":   str,
                "label":      "positive" | "neutral" | "negative",
                "score":      +1.0 | 0.0 | -1.0,
                "confidence": float  (0.0 - 1.0)
            }
        """
        inputs = self.tokenizer(
            headline,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = self.model(**inputs).logits

        probs = softmax(logits, dim=-1).cpu().numpy()[0]
        pred_idx    = int(np.argmax(probs))
        label       = self.label_map[pred_idx]
        confidence  = float(probs[pred_idx])

        return {
            "headline":   headline,
            "label":      label,
            "score":      self.score_map[label],
            "confidence": confidence
        }

    def analyze_batch(self, headlines: list[str]) -> dict:
        """
        Analyzes a list of headlines.

        Returns aggregate:
            {
                "results":         list of per-headline dicts,
                "avg_score":       float  (-1.0 to +1.0),
                "sentiment_label": "Positive" | "Neutral" | "Negative",
                "positive_count":  int,
                "neutral_count":   int,
                "negative_count":  int
            }
        """
        if not headlines:
            return {
                "results":         [],
                "avg_score":       0.0,
                "sentiment_label": "Neutral",
                "positive_count":  0,
                "neutral_count":   0,
                "negative_count":  0
            }

        results = [self.analyze_headline(h) for h in headlines]
        scores  = [r["score"] for r in results]
        avg     = float(np.mean(scores))

        if avg > 0.1:
            label = "Positive"
        elif avg < -0.1:
            label = "Negative"
        else:
            label = "Neutral"

        return {
            "results":         results,
            "avg_score":       avg,
            "sentiment_label": label,
            "positive_count":  sum(1 for r in results if r["label"] == "positive"),
            "neutral_count":   sum(1 for r in results if r["label"] == "neutral"),
            "negative_count":  sum(1 for r in results if r["label"] == "negative"),
        }


def fetch_news_headlines(yahoo_ticker, max_headlines=10):
    """
    Fetch news headlines from Yahoo Finance.
    Compatible with latest yfinance versions.
    """

    try:
        ticker = yf.Ticker(yahoo_ticker)
        news = ticker.news

        if not news:
            print(f"⚠️ No news found for {yahoo_ticker}")
            return []

        headlines = []

        for item in news[:max_headlines]:

            # New yfinance structure
            if "content" in item:
                content = item["content"]

                title = (
                    content.get("title")
                    or content.get("headline")
                    or content.get("summary")
                    or ""
                )

            # Old yfinance structure
            else:
                title = (
                    item.get("title")
                    or item.get("headline")
                    or item.get("summary")
                    or ""
                )

            if title:
                headlines.append(title.strip())

        print(f"✅ Fetched {len(headlines)} headlines")

        return headlines

    except Exception as e:
        print(e)
        return []

In [ ]:
def compute_hybrid_score(
    prob_up:       float,
    sentiment_avg: float,
    w_model:       float = WEIGHT_MODEL,
    w_sentiment:   float = WEIGHT_SENTIMENT
) -> dict:
    """
    Combines model probability and FinBERT sentiment.

    Formula:
        combined_score = w_model * prob_up
                       + w_sentiment * ((sentiment_avg + 1) / 2)

    sentiment_avg is in [-1, +1]; normalised to [0, 1] via (avg + 1) / 2.

    Returns dict with all component values and final recommendation.
    """
    sentiment_normalised = (sentiment_avg + 1.0) / 2.0
    combined_score = w_model * prob_up + w_sentiment * sentiment_normalised

    # Clamp to [0, 1]
    combined_score = float(np.clip(combined_score, 0.0, 1.0))

    if combined_score >= BUY_THRESHOLD:
        recommendation = "BUY"
    elif combined_score <= SELL_THRESHOLD:
        recommendation = "SELL"
    else:
        recommendation = "HOLD"

    return {
        "prob_up":               prob_up,
        "sentiment_avg":         sentiment_avg,
        "sentiment_normalised":  sentiment_normalised,
        "combined_score":        combined_score,
        "recommendation":        recommendation
    }


def build_reasoning(
    stock_name:      str,
    decision:        dict,
    sentiment_result: dict,
    pred_direction:  int,
    current_price:   float
) -> str:
    """
    Generates a human-readable reasoning paragraph.
    """
    rec   = decision["recommendation"]
    prob  = decision["prob_up"] * 100.0
    cs    = decision["combined_score"]
    senti = sentiment_result["sentiment_label"]
    s_avg = decision["sentiment_avg"]

    direction_str = "UP ↑" if pred_direction == 1 else "DOWN ↓"

    lines = []

    # Technical reasoning
    if decision["prob_up"] >= 0.60:
        lines.append(
            f"The XGBoost model assigns a strong {prob:.1f}% probability of "
            f"upward price movement over the next 5 trading days for {stock_name}, "
            f"suggesting a technically bullish short-term setup."
        )
    elif decision["prob_up"] >= 0.50:
        lines.append(
            f"The XGBoost model gives a moderate {prob:.1f}% probability of upward "
            f"movement for {stock_name}, indicating a mildly positive technical signal."
        )
    else:
        lines.append(
            f"The XGBoost model assigns only {prob:.1f}% probability of upward "
            f"movement for {stock_name}, reflecting bearish or uncertain technical conditions."
        )

    # Sentiment reasoning
    pos = sentiment_result.get("positive_count", 0)
    neg = sentiment_result.get("negative_count", 0)
    neu = sentiment_result.get("neutral_count", 0)
    total_news = pos + neg + neu

    if total_news == 0:
        lines.append(
            "No recent news headlines were available; "
            "sentiment defaults to Neutral (score = 0.0)."
        )
    else:
        lines.append(
            f"News sentiment is {senti} (score: {s_avg:+.2f}) based on {total_news} headlines "
            f"[Positive: {pos}, Neutral: {neu}, Negative: {neg}]."
        )

    # Hybrid combined
    lines.append(
        f"The hybrid combined score of {cs:.4f} "
        f"(70% model probability + 30% normalised sentiment) "
        f"falls {'above' if rec == 'BUY' else 'below' if rec == 'SELL' else 'within'} "
        f"{'the BUY threshold of ' + str(BUY_THRESHOLD) if rec == 'BUY' else 'the SELL threshold of ' + str(SELL_THRESHOLD) if rec == 'SELL' else 'the HOLD range (0.40 – 0.70)'}."
    )

    # Final recommendation rationale
    if rec == "BUY":
        lines.append(
            f"Both technical momentum and market sentiment support a BUY signal. "
            f"Consider initiating or adding to a long position in {stock_name} "
            f"near the current price of ₹{current_price:,.2f}."
        )
    elif rec == "SELL":
        lines.append(
            f"Weak technical probability and/or negative sentiment support a SELL signal. "
            f"Consider reducing exposure or avoiding new long positions in {stock_name} "
            f"at the current price of ₹{current_price:,.2f}."
        )
    else:
        lines.append(
            f"Mixed signals from the model and/or sentiment suggest a HOLD stance. "
            f"Monitor {stock_name} closely for confirmation before taking new positions "
            f"at ₹{current_price:,.2f}."
        )

    return " ".join(lines)


print("✅ Hybrid decision engine defined.")

✅ Hybrid decision engine defined.


In [ ]:
def render_report(
    stock_name:        str,
    yahoo_ticker:      str,
    current_price:     float,
    prob_up:           float,
    pred_direction:    int,
    sentiment_result:  dict,
    decision:          dict,
    reasoning:         str,
    news_headlines:    list[str]
):
    """
    Prints the full AI Stock Analysis Report to console
    AND renders an HTML card in Colab output.
    """
    rec  = decision["recommendation"]
    cs   = decision["combined_score"]
    s_avg = decision["sentiment_avg"]

    dir_str   = "UP ↑" if pred_direction == 1 else "DOWN ↓"
    conf_pct  = prob_up * 100.0
    senti_lbl = sentiment_result["sentiment_label"]

    # ---- Colour coding ----
    rec_color = {"BUY": "#00c853", "HOLD": "#ff6f00", "SELL": "#d50000"}[rec]
    dir_color = "#00c853" if pred_direction == 1 else "#d50000"

    # ---- Console Output ----
    sep = "=" * 55
    print(f"\n{sep}")
    print("    AI STOCK ANALYSIS REPORT")
    print(sep)
    print(f"  Stock          : {stock_name}  ({yahoo_ticker})")
    print(f"  Current Price  : ₹{current_price:,.2f}")
    print(sep)
    print("  TECHNICAL ANALYSIS")
    print(f"    Probability Up      : {conf_pct:.2f}%")
    print(f"    Predicted Direction : {dir_str}")
    print(sep)
    print("  NEWS SENTIMENT  (FinBERT)")
    print(f"    Sentiment Label     : {senti_lbl}")
    print(f"    Sentiment Score     : {s_avg:+.4f}")
    print(f"    Headlines Analysed  : {len(news_headlines)}")
    if news_headlines:
        print("    Latest Headlines:")
        for i, h in enumerate(news_headlines[:5], 1):
            print(f"      {i}. {h[:80]}{'...' if len(h) > 80 else ''}")
    print(sep)
    print("  COMBINED ANALYSIS")
    print(f"    Model Prob (70%)    : {prob_up * WEIGHT_MODEL:.4f}")
    print(f"    Sentiment Norm(30%) : {decision['sentiment_normalised'] * WEIGHT_SENTIMENT:.4f}")
    print(f"    Combined Score      : {cs:.4f}")
    print(sep)
    print(f"  FINAL RECOMMENDATION  →  {rec}")
    print(sep)
    print("  REASONING")
    # Word-wrap reasoning at 52 chars
    words = reasoning.split()
    line  = "    "
    for word in words:
        if len(line) + len(word) + 1 > 56:
            print(line)
            line = "    " + word + " "
        else:
            line += word + " "
    if line.strip():
        print(line)
    print(sep)

    # ---- HTML Output ----
    news_rows_html = ""
    for h in news_headlines[:5]:
        news_rows_html += f"<li style='margin:4px 0;'>{h[:100]}{'...' if len(h)>100 else ''}</li>"
    if not news_rows_html:
        news_rows_html = "<li><i>No headlines available</i></li>"

    detail_rows = [
        ("positive_count",  "Positive Headlines"),
        ("neutral_count",   "Neutral Headlines"),
        ("negative_count",  "Negative Headlines"),
    ]
    senti_detail_html = "".join(
        f"<tr><td style='padding:3px 8px;color:#aaa;'>{label}</td>"
        f"<td style='padding:3px 8px;text-align:right;'>{sentiment_result.get(key,0)}</td></tr>"
        for key, label in detail_rows
    )

    html = f"""
    <div style="
        font-family: 'Segoe UI', Arial, sans-serif;
        background: #1a1a2e;
        color: #e0e0e0;
        border-radius: 12px;
        padding: 28px 32px;
        max-width: 720px;
        margin: 20px auto;
        box-shadow: 0 4px 24px rgba(0,0,0,0.5);
    ">
        <!-- Header -->
        <div style="border-bottom:1px solid #333;padding-bottom:14px;margin-bottom:18px;">
            <h2 style="margin:0;color:#90caf9;letter-spacing:1px;">🤖 AI STOCK ANALYSIS REPORT</h2>
            <div style="margin-top:8px;font-size:1.1em;">
                <span style="color:#fff;font-weight:bold;">{stock_name}</span>
                <span style="color:#888;margin-left:10px;">({yahoo_ticker})</span>
                <span style="float:right;color:#ffd54f;font-size:1.2em;font-weight:bold;">
                    ₹{current_price:,.2f}
                </span>
            </div>
        </div>

        <!-- Technical -->
        <div style="margin-bottom:16px;">
            <h3 style="color:#80cbc4;margin:0 0 10px 0;">📊 TECHNICAL ANALYSIS</h3>
            <table style="width:100%;border-collapse:collapse;">
                <tr>
                    <td style="padding:4px 8px;color:#aaa;">Probability Up (5-day)</td>
                    <td style="padding:4px 8px;text-align:right;font-weight:bold;color:#fff;">
                        {conf_pct:.2f}%
                    </td>
                </tr>
                <tr>
                    <td style="padding:4px 8px;color:#aaa;">Predicted Direction</td>
                    <td style="padding:4px 8px;text-align:right;font-weight:bold;color:{dir_color};">
                        {dir_str}
                    </td>
                </tr>
            </table>
        </div>

        <!-- Sentiment -->
        <div style="margin-bottom:16px;">
            <h3 style="color:#80cbc4;margin:0 0 10px 0;">📰 NEWS SENTIMENT (FinBERT)</h3>
            <table style="width:100%;border-collapse:collapse;">
                <tr>
                    <td style="padding:4px 8px;color:#aaa;">Sentiment Label</td>
                    <td style="padding:4px 8px;text-align:right;font-weight:bold;color:#fff;">
                        {senti_lbl}
                    </td>
                </tr>
                <tr>
                    <td style="padding:4px 8px;color:#aaa;">Sentiment Score</td>
                    <td style="padding:4px 8px;text-align:right;font-weight:bold;color:#fff;">
                        {s_avg:+.4f}
                    </td>
                </tr>
                {senti_detail_html}
            </table>
            <div style="margin-top:8px;font-size:0.85em;color:#aaa;">
                <b>Latest Headlines:</b>
                <ul style="margin:4px 0 0 0;padding-left:18px;">{news_rows_html}</ul>
            </div>
        </div>

        <!-- Combined -->
        <div style="margin-bottom:16px;">
            <h3 style="color:#80cbc4;margin:0 0 10px 0;">⚖️ COMBINED ANALYSIS</h3>
            <table style="width:100%;border-collapse:collapse;">
                <tr>
                    <td style="padding:4px 8px;color:#aaa;">Model Score (70%)</td>
                    <td style="padding:4px 8px;text-align:right;">{prob_up * WEIGHT_MODEL:.4f}</td>
                </tr>
                <tr>
                    <td style="padding:4px 8px;color:#aaa;">Sentiment Score (30%)</td>
                    <td style="padding:4px 8px;text-align:right;">
                        {decision['sentiment_normalised'] * WEIGHT_SENTIMENT:.4f}
                    </td>
                </tr>
                <tr style="border-top:1px solid #333;">
                    <td style="padding:6px 8px;color:#fff;font-weight:bold;">Combined Score</td>
                    <td style="padding:6px 8px;text-align:right;font-weight:bold;font-size:1.1em;color:#ffd54f;">
                        {cs:.4f}
                    </td>
                </tr>
            </table>
        </div>

        <!-- Recommendation -->
        <div style="
            background:{rec_color};
            color:#000;
            border-radius:8px;
            padding:14px 20px;
            text-align:center;
            font-size:1.6em;
            font-weight:900;
            letter-spacing:3px;
            margin-bottom:16px;
        ">
            FINAL RECOMMENDATION: {rec}
        </div>

        <!-- Reasoning -->
        <div style="background:#0d0d1a;border-radius:8px;padding:14px 18px;">
            <h3 style="color:#80cbc4;margin:0 0 8px 0;">💡 REASONING</h3>
            <p style="margin:0;line-height:1.6;font-size:0.95em;color:#ccc;">{reasoning}</p>
        </div>
    </div>
    """

    display(HTML(html))


print("✅ Report renderer defined.")

✅ Report renderer defined.


In [ ]:
print("📁 Please upload the three model artifact files:")
print("   • portfolio_models_dict.pkl")
print("   • feature_columns.pkl")
print("   • stock_encoder.pkl\n")

uploaded_artifacts = files.upload()

# ---- Validate uploaded artifact files ----
required_artifacts = {
    "portfolio_models_dict.pkl": None,
    "feature_columns.pkl":       None,
    "stock_encoder.pkl":         None
}

missing_artifacts = []
for artifact_name in required_artifacts:
    if artifact_name in uploaded_artifacts:
        required_artifacts[artifact_name] = uploaded_artifacts[artifact_name]
        print(f"   ✅ Found: {artifact_name}")
    else:
        missing_artifacts.append(artifact_name)
        print(f"   ❌ Missing: {artifact_name}")

if missing_artifacts:
    raise FileNotFoundError(
        f"The following required artifact files were not uploaded:\n"
        f"{missing_artifacts}\n"
        "Please re-run this cell and upload all three files."
    )

# ---- Load artifacts ----
print("\n📦 Loading model artifacts ...")

portfolio_models = joblib.load(io.BytesIO(required_artifacts["portfolio_models_dict.pkl"]))
feature_columns  = joblib.load(io.BytesIO(required_artifacts["feature_columns.pkl"]))
stock_encoder    = joblib.load(io.BytesIO(required_artifacts["stock_encoder.pkl"]))

print(f"   ✅ portfolio_models loaded  : {len(portfolio_models)} models")
print(f"   ✅ feature_columns loaded   : {len(feature_columns)} features")
print(f"   ✅ stock_encoder loaded     : {list(stock_encoder.classes_[:5])} ...")
print("\n✅ All model artifacts loaded successfully.")

📁 Please upload the three model artifact files:
   • portfolio_models_dict.pkl
   • feature_columns.pkl
   • stock_encoder.pkl



Saving portfolio_models_dict.pkl to portfolio_models_dict.pkl
Saving stock_encoder.pkl to stock_encoder.pkl
Saving feature_columns.pkl to feature_columns.pkl
   ✅ Found: portfolio_models_dict.pkl
   ✅ Found: feature_columns.pkl
   ✅ Found: stock_encoder.pkl

📦 Loading model artifacts ...
   ✅ portfolio_models loaded  : 49 models
   ✅ feature_columns loaded   : 97 features
   ✅ stock_encoder loaded     : ['ADANIENT.NS', 'ADANIPORTS.NS', 'APOLLOHOSP.NS', 'ASIANPAINT.NS', 'AXISBANK.NS'] ...

✅ All model artifacts loaded successfully.


In [ ]:
print("📁 Please upload your stock CSV file.")
print("   Filename format: STOCKNAME.csv  (e.g. RELIANCE.csv, TCS.csv)\n")

uploaded_csv = files.upload()

if not uploaded_csv:
    raise ValueError("❌ No CSV file was uploaded. Please re-run this cell.")

csv_filename = list(uploaded_csv.keys())[0]
csv_content  = uploaded_csv[csv_filename]

print(f"\n   📄 Uploaded file : {csv_filename}")
print(f"   📦 File size      : {len(csv_content):,} bytes")

# ---- Detect stock symbol ----
try:
    raw_symbol, yahoo_ticker = detect_stock_symbol(csv_filename)
    print(f"\n   ✅ Detected stock symbol : {raw_symbol}")
    print(f"   ✅ Yahoo Finance ticker  : {yahoo_ticker}")
except ValueError as e:
    print(str(e))
    # Allow manual override
    print("\n   ⚙️  Manual Override: Enter the ticker symbol (e.g. RELIANCE for RELIANCE.NS):")
    manual_input = input("   Ticker (without .NS): ").strip().upper()
    if manual_input in KNOWN_STOCKS:
        raw_symbol, yahoo_ticker = manual_input, KNOWN_STOCKS[manual_input]
        print(f"   ✅ Using manually entered ticker: {yahoo_ticker}")
    else:
        raise ValueError(
            f"Manual ticker '{manual_input}' is not in the known stocks list."
        )

# ---- Validate model availability ----
if yahoo_ticker not in portfolio_models:
    raise KeyError(
        f"❌ No trained model found for '{yahoo_ticker}'.\n"
        f"   Available models: {sorted(portfolio_models.keys())}"
    )

print(f"\n   ✅ Trained model found for : {yahoo_ticker}")

# ---- Parse and validate CSV ----
try:

    raw_df = pd.read_csv(io.BytesIO(csv_content))

    print(
        f"   ✅ CSV parsed              : "
        f"{raw_df.shape[0]} rows × {raw_df.shape[1]} columns"
    )

    # --------------------------------------
    # Handle Yahoo Finance exports
    # --------------------------------------

    if "Date" not in raw_df.columns:

        first_col = str(raw_df.columns[0])

        if "date" in first_col.lower():

            raw_df.rename(
                columns={raw_df.columns[0]: "Date"},
                inplace=True
            )

    # Remove metadata rows
    raw_df = raw_df[
        ~raw_df.iloc[:, 0].astype(str).isin(
            ["Ticker", "Date", ""]
        )
    ]

    # Keep rows containing dates
    raw_df = raw_df[
        raw_df.iloc[:, 0].astype(str).str.contains(
            r"\d",
            regex=True,
            na=False
        )
    ]

    raw_df.reset_index(
        drop=True,
        inplace=True
    )

    # Rename columns if Yahoo export changed them
    rename_map = {}

    for col in raw_df.columns:

        col_name = str(col).strip().lower()

        if "date" in col_name:
            rename_map[col] = "Date"

        elif "open" == col_name:
            rename_map[col] = "Open"

        elif "high" == col_name:
            rename_map[col] = "High"

        elif "low" == col_name:
            rename_map[col] = "Low"

        elif "close" == col_name:
            rename_map[col] = "Close"

        elif "volume" == col_name:
            rename_map[col] = "Volume"

    raw_df.rename(
        columns=rename_map,
        inplace=True
    )

    required_cols = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    missing_cols = [
        c for c in required_cols
        if c not in raw_df.columns
    ]

    if missing_cols:

        print("\nColumns Found:")
        print(list(raw_df.columns))

        raise ValueError(
            f"❌ Missing required columns: {missing_cols}"
        )

    # Convert date
    raw_df["Date"] = pd.to_datetime(
        raw_df["Date"],
        errors="coerce"
    )

    raw_df = raw_df[
        raw_df["Date"].notna()
    ]

    # Convert numeric columns
    for col in [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]:

        raw_df[col] = pd.to_numeric(
            raw_df[col],
            errors="coerce"
        )

    raw_df.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ],
        inplace=True
    )

    raw_df.sort_values(
        "Date",
        inplace=True
    )

    raw_df.reset_index(
        drop=True,
        inplace=True
    )

    csv_df = raw_df.copy()

    print(
        f"   ✅ CSV validated           : "
        f"{len(csv_df)} usable rows"
    )

    print(
        f"   📅 Date range             : "
        f"{csv_df['Date'].min().date()} → "
        f"{csv_df['Date'].max().date()}"
    )

except Exception as e:

    print("\n❌ CSV Validation Failed")
    print(str(e))

    print("\nDetected Columns:")
    print(raw_df.columns.tolist())

    print("\nFirst 10 Rows:")
    print(raw_df.head(10))

    raise

print("\n✅ CSV file validated and ready for feature engineering.")

📁 Please upload your stock CSV file.
   Filename format: STOCKNAME.csv  (e.g. RELIANCE.csv, TCS.csv)



Saving TCS_history.csv to TCS_history (1).csv

   📄 Uploaded file : TCS_history (1).csv
   📦 File size      : 544,544 bytes

   ✅ Detected stock symbol : TCS
   ✅ Yahoo Finance ticker  : TCS.NS

   ✅ Trained model found for : TCS.NS
   ✅ CSV parsed              : 5935 rows × 6 columns
   ✅ CSV validated           : 5935 usable rows
   📅 Date range             : 2002-08-12 → 2026-07-08

✅ CSV file validated and ready for feature engineering.


In [ ]:
print(f"\n⚙️  Engineering features for {yahoo_ticker} ...")

# Apply create_stationary_features
try:
    csv_df["Date"] = pd.to_datetime(csv_df["Date"]).dt.tz_localize(None)
    engineered_df  = create_stationary_features(csv_df)
except Exception as e:
    raise RuntimeError(f"❌ Feature engineering failed: {e}")

# Encode Stock_ID using the saved LabelEncoder
try:
    engineered_df["Stock_ID"] = stock_encoder.transform([yahoo_ticker])[0]
    print(f"   ✅ Stock_ID encoded as: {engineered_df['Stock_ID'].iloc[0]}")
except ValueError:
    # Stock was not seen during training encoding → assign 0 with warning
    print(f"   ⚠️  '{yahoo_ticker}' not found in LabelEncoder. Assigning Stock_ID = 0.")
    engineered_df["Stock_ID"] = 0

# Replace infinities → NaN then drop NaNs
engineered_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Keep only non-NaN rows (excluding the last 5 rows which have no target anyway)
valid_df = engineered_df.dropna(subset=feature_columns).copy()

if len(valid_df) == 0:
    raise ValueError(
        "❌ No valid rows after feature engineering and NaN removal.\n"
        "   The CSV may be too short or have too many missing values.\n"
        "   Minimum 60+ rows recommended."
    )

print(f"   ✅ Valid rows after feature engineering: {len(valid_df)}")

# ---- Select the LATEST row for prediction ----
latest_row = valid_df.sort_values("Date").iloc[[-1]].copy()
current_price_csv = float(latest_row["Close"].values[0])
latest_date = latest_row["Date"].values[0]

print(f"   📅 Predicting using data as of : {pd.Timestamp(latest_date).date()}")
print(f"   💰 Last Close in CSV            : ₹{current_price_csv:,.2f}")

# ---- Verify all features are present ----
missing_features = [f for f in feature_columns if f not in latest_row.columns]
if missing_features:
    raise ValueError(
        f"❌ The following required features are missing after engineering:\n"
        f"{missing_features}"
    )

X_input = latest_row[feature_columns]
print(f"\n✅ Feature matrix ready: shape {X_input.shape}")


⚙️  Engineering features for TCS.NS ...
   ✅ Stock_ID encoded as: 43
   ✅ Valid rows after feature engineering: 5337
   📅 Predicting using data as of : 2026-06-26
   💰 Last Close in CSV            : ₹2,094.70

✅ Feature matrix ready: shape (1, 97)


In [ ]:
print(f"\n🤖 Running XGBoost prediction for {yahoo_ticker} ...")

model = portfolio_models[yahoo_ticker]

try:
    pred_label    = int(model.predict(X_input)[0])
    pred_proba    = model.predict_proba(X_input)[0]
    prob_up       = float(pred_proba[1])
    prob_down     = float(pred_proba[0])
except Exception as e:
    raise RuntimeError(f"❌ Model prediction failed: {e}")

direction_str = "UP ↑ (Bullish)" if pred_label == 1 else "DOWN ↓ (Bearish)"
confidence    = max(prob_up, prob_down) * 100.0

print(f"   ✅ Predicted Direction  : {direction_str}")
print(f"   ✅ Probability UP       : {prob_up * 100:.2f}%")
print(f"   ✅ Probability DOWN     : {prob_down * 100:.2f}%")
print(f"   ✅ Prediction Confidence: {confidence:.2f}%")


🤖 Running XGBoost prediction for TCS.NS ...
   ✅ Predicted Direction  : UP ↑ (Bullish)
   ✅ Probability UP       : 80.91%
   ✅ Probability DOWN     : 19.09%
   ✅ Prediction Confidence: 80.91%


In [ ]:
print("\n🧠 Loading FinBERT model (this may take ~30 seconds the first time) ...")

try:
    finbert = FinBERTAnalyzer(model_name=FINBERT_MODEL)
except Exception as e:
    raise RuntimeError(f"❌ FinBERT failed to load: {e}")

print(f"\n📰 Fetching latest news headlines for {yahoo_ticker} ...")
headlines = fetch_news_headlines(yahoo_ticker, max_headlines=10)

print(f"\n🔍 Analysing {len(headlines)} headlines with FinBERT ...")

if headlines:
    sentiment_result = finbert.analyze_batch(headlines)
    print(f"   ✅ Sentiment Label      : {sentiment_result['sentiment_label']}")
    print(f"   ✅ Average Score        : {sentiment_result['avg_score']:+.4f}")
    print(f"   📊 Positive / Neutral / Negative : "
          f"{sentiment_result['positive_count']} / "
          f"{sentiment_result['neutral_count']} / "
          f"{sentiment_result['negative_count']}")

    print("\n   Headline-level results:")
    for i, res in enumerate(sentiment_result["results"], 1):
        emoji = {"positive": "🟢", "neutral": "🟡", "negative": "🔴"}[res["label"]]
        print(f"   {i:2}. {emoji} [{res['label'].upper():8s}] "
              f"(conf: {res['confidence']:.2f}) "
              f"{res['headline'][:65]}{'...' if len(res['headline'])>65 else ''}")
else:
    # Fallback: neutral sentiment when no news available
    sentiment_result = {
        "results":         [],
        "avg_score":       0.0,
        "sentiment_label": "Neutral",
        "positive_count":  0,
        "neutral_count":   0,
        "negative_count":  0
    }
    print("   ⚠️  No headlines available. Defaulting to Neutral sentiment (score = 0.0).")


🧠 Loading FinBERT model (this may take ~30 seconds the first time) ...
   Loading FinBERT from 'ProsusAI/finbert' ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

   FinBERT loaded on device: cpu

📰 Fetching latest news headlines for TCS.NS ...
✅ Fetched 10 headlines

🔍 Analysing 10 headlines with FinBERT ...
   ✅ Sentiment Label      : Neutral
   ✅ Average Score        : +0.0000
   📊 Positive / Neutral / Negative : 5 / 0 / 5

   Headline-level results:
    1. 🟢 [POSITIVE] (conf: 0.83) India's TCS rises after quarterly revenue beat
    2. 🟢 [POSITIVE] (conf: 0.84) Indian shares set to open higher; TCS in focus after revenue beat
    3. 🟢 [POSITIVE] (conf: 0.95) Tata Consultancy Services Ltd (BOM:532540) Q1 2027 Earnings Call ...
    4. 🟢 [POSITIVE] (conf: 0.81) India's TCS tops revenue estimates on weak rupee, banking boost
    5. 🟢 [POSITIVE] (conf: 0.79) Indian shares stage partial rebound; Mideast tensions cap gains
    6. 🔴 [NEGATIVE] (conf: 0.97) Indian IT firms face muted Q1 as AI shift, weak demand weigh
    7. 🔴 [NEGATIVE] (conf: 0.59) Investors Looking for Shelter From AI Storm Are Turning to India
    8. 🔴 [NEGATIVE] (conf: 0.97) China

In [ ]:
print("\n⚖️  Computing hybrid combined score ...")

decision = compute_hybrid_score(
    prob_up       = prob_up,
    sentiment_avg = sentiment_result["avg_score"],
    w_model       = WEIGHT_MODEL,
    w_sentiment   = WEIGHT_SENTIMENT
)

print(f"   Model contribution     : {prob_up:.4f} × {WEIGHT_MODEL} = {prob_up * WEIGHT_MODEL:.4f}")
print(f"   Sentiment contribution : {decision['sentiment_normalised']:.4f} × "
      f"{WEIGHT_SENTIMENT} = {decision['sentiment_normalised'] * WEIGHT_SENTIMENT:.4f}")
print(f"   Combined Score         : {decision['combined_score']:.4f}")
print(f"   ✅ RECOMMENDATION      : {decision['recommendation']}")


⚖️  Computing hybrid combined score ...
   Model contribution     : 0.8091 × 0.7 = 0.5663
   Sentiment contribution : 0.5000 × 0.3 = 0.1500
   Combined Score         : 0.7163
   ✅ RECOMMENDATION      : BUY


In [ ]:
print(f"\n💹 Fetching live current price for {yahoo_ticker} ...")

current_price = safe_get_current_price(yahoo_ticker, fallback_close=current_price_csv)

if current_price == current_price_csv:
    print(f"   ⚠️  Live price unavailable. Using last CSV close: ₹{current_price:,.2f}")
else:
    print(f"   ✅ Live price fetched: ₹{current_price:,.2f}")

# Build reasoning paragraph
reasoning = build_reasoning(
    stock_name       = yahoo_ticker,
    decision         = decision,
    sentiment_result = sentiment_result,
    pred_direction   = pred_label,
    current_price    = current_price
)

print("\n✅ Reasoning generated.")


💹 Fetching live current price for TCS.NS ...
   ✅ Live price fetched: ₹2,076.80

✅ Reasoning generated.


In [ ]:
render_report(
    stock_name        = yahoo_ticker,
    yahoo_ticker      = yahoo_ticker,
    current_price     = current_price,
    prob_up           = prob_up,
    pred_direction    = pred_label,
    sentiment_result  = sentiment_result,
    decision          = decision,
    reasoning         = reasoning,
    news_headlines    = headlines
)


    AI STOCK ANALYSIS REPORT
  Stock          : TCS.NS  (TCS.NS)
  Current Price  : ₹2,076.80
  TECHNICAL ANALYSIS
    Probability Up      : 80.91%
    Predicted Direction : UP ↑
  NEWS SENTIMENT  (FinBERT)
    Sentiment Label     : Neutral
    Sentiment Score     : +0.0000
    Headlines Analysed  : 10
    Latest Headlines:
      1. India's TCS rises after quarterly revenue beat
      2. Indian shares set to open higher; TCS in focus after revenue beat
      3. Tata Consultancy Services Ltd (BOM:532540) Q1 2027 Earnings Call Highlights: Str...
      4. India's TCS tops revenue estimates on weak rupee, banking boost
      5. Indian shares stage partial rebound; Mideast tensions cap gains
  COMBINED ANALYSIS
    Model Prob (70%)    : 0.5663
    Sentiment Norm(30%) : 0.1500
    Combined Score      : 0.7163
  FINAL RECOMMENDATION  →  BUY
  REASONING
    The XGBoost model assigns a strong 80.9% 
    probability of upward price movement over the next 
    5 trading days for TCS.NS, suggesti

Probability Up (5-day),80.91%
Predicted Direction,UP ↑
Sentiment Label,Neutral
Sentiment Score,+0.0000
Positive Headlines,5
Neutral Headlines,0
Negative Headlines,5
Model Score (70%),0.5663
Sentiment Score (30%),0.1500
Combined Score,0.7163


In [ ]:
def run_batch_analysis(
    portfolio_models: dict,
    feature_columns:  list,
    stock_encoder:    LabelEncoder,
    finbert:          FinBERTAnalyzer
) -> pd.DataFrame:
    """
    Upload and analyse multiple stock CSVs in one batch.
    Returns a summary DataFrame with all recommendations.
    """
    print("📁 Upload multiple stock CSV files for batch analysis ...\n")
    uploaded_files = files.upload()

    if not uploaded_files:
        print("❌ No files uploaded.")
        return pd.DataFrame()

    batch_results = []

    for fname, fdata in uploaded_files.items():
        print(f"\n{'='*55}")
        print(f"  Processing: {fname}")
        print(f"{'='*55}")

        try:
            # Detect symbol
            try:
                raw_sym, ticker = detect_stock_symbol(fname)
            except ValueError as e:
                print(f"  ❌ Symbol detection failed: {e}")
                batch_results.append({"File": fname, "Ticker": "UNKNOWN",
                                      "Error": "Symbol not recognised"})
                continue

            # Check model
            if ticker not in portfolio_models:
                print(f"  ❌ No model for {ticker}")
                batch_results.append({"File": fname, "Ticker": ticker,
                                      "Error": "No trained model"})
                continue

            # Parse CSV
            raw = pd.read_csv(io.BytesIO(fdata))
            df  = validate_csv_columns(raw)

            # Feature engineering
            df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
            eng_df = create_stationary_features(df)

            try:
                eng_df["Stock_ID"] = stock_encoder.transform([ticker])[0]
            except ValueError:
                eng_df["Stock_ID"] = 0

            eng_df.replace([np.inf, -np.inf], np.nan, inplace=True)
            valid = eng_df.dropna(subset=feature_columns)

            if len(valid) == 0:
                print(f"  ❌ No valid rows after feature engineering.")
                batch_results.append({"File": fname, "Ticker": ticker,
                                      "Error": "No valid feature rows"})
                continue

            latest     = valid.sort_values("Date").iloc[[-1]]
            last_close = float(latest["Close"].values[0])
            X_in       = latest[feature_columns]

            # Predict
            model  = portfolio_models[ticker]
            p_up   = float(model.predict_proba(X_in)[0][1])
            p_dir  = int(model.predict(X_in)[0])

            # Sentiment
            news_h = fetch_news_headlines(ticker, max_headlines=10)
            sent_r = finbert.analyze_batch(news_h) if news_h else {
                "avg_score": 0.0, "sentiment_label": "Neutral",
                "positive_count": 0, "neutral_count": 0, "negative_count": 0,
                "results": []
            }

            # Decision
            dec = compute_hybrid_score(p_up, sent_r["avg_score"])

            # Live price
            live_px = safe_get_current_price(ticker, last_close)

            batch_results.append({
                "File":              fname,
                "Ticker":            ticker,
                "Current_Price":     live_px,
                "Prob_Up_%":         round(p_up * 100, 2),
                "Pred_Direction":    "UP" if p_dir == 1 else "DOWN",
                "Sentiment_Label":   sent_r["sentiment_label"],
                "Sentiment_Score":   round(sent_r["avg_score"], 4),
                "Combined_Score":    round(dec["combined_score"], 4),
                "Recommendation":    dec["recommendation"],
                "Error":             None
            })

            print(f"  ✅ {ticker} → {dec['recommendation']} "
                  f"(score: {dec['combined_score']:.4f}, prob_up: {p_up*100:.1f}%)")

        except Exception as e:
            print(f"  ❌ Error processing {fname}: {e}")
            batch_results.append({"File": fname, "Ticker": "ERROR", "Error": str(e)})

    summary_df = pd.DataFrame(batch_results)

    if not summary_df.empty:
        print(f"\n{'='*55}")
        print("  BATCH ANALYSIS SUMMARY")
        print(f"{'='*55}")
        display(summary_df)

        # Save results
        summary_df.to_csv("batch_analysis_results.csv", index=False)
        print("\n💾 Results saved to: batch_analysis_results.csv")
        files.download("batch_analysis_results.csv")

    return summary_df



batch_summary = run_batch_analysis(

    portfolio_models, feature_columns, stock_encoder, finbert
)

print("✅ Batch analysis function ready.")
print("   To run: uncomment the last 4 lines in this cell.")

📁 Upload multiple stock CSV files for batch analysis ...



KeyboardInterrupt: 

In [ ]:
import os

for f in [
    "portfolio_models_dict.pkl",
    "feature_columns.pkl",
    "stock_encoder.pkl"
]:
    if os.path.exists(f):
        os.remove(f)

In [ ]:
import yfinance as yf

ticker = yf.Ticker("WIPRO.NS")

print(ticker.news)
print(len(ticker.news))

In [ ]:
for stock in [
    "AAPL",
    "MSFT",
    "TSLA",
    "RELIANCE.NS",
    "INFY.NS",
    "WIPRO.NS"
]:
    t = yf.Ticker(stock)
    print(stock, len(t.news))